In [2]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   National Energy Digital Twin (NEDT) — Ontology Diagram Generator                  ║
║   Generates 9 SVG diagrams embedded in a single HTML file                  ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  QUICK START (Mac / VS Code)                                                 ║
║  ─────────────────────────                                                   ║
║  1. Open this file in VS Code                                                ║
║  2. Open the integrated terminal:  View → Terminal  (or  Ctrl + `)           ║
║  3. Run:   python3 gen_diagrams.py                                           ║
║  4. A file called  ontology_diagrams.html  appears in the same folder       ║
║  5. Double-click it to open in Safari / Chrome — use the download buttons   ║
║                                                                              ║
║  No pip installs needed — uses only Python's standard library.              ║
║                                                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  HOW TO EDIT A DIAGRAM                                                       ║
║  ─────────────────────                                                       ║
║  Each figure is a function: fig0() … fig8()                                 ║
║  Inside each function, build up a string  b  using these helpers:           ║
║                                                                              ║
║  box(x, y, w, h, fill, stroke, "Title", "optional subtitle")               ║
║      → coloured rectangle with centred text                                 ║
║      → x, y = top-left corner;  w = width;  h = height                     ║
║      → use C["teal_f"] for fill, C["teal_s"] for stroke                    ║
║      → 36px tall for a single-line box, 52px for a two-line box             ║
║                                                                              ║
║  gbox(x, y, w, h, "Title", "optional subtitle")                             ║
║      → shorthand for a grey data-type / helper box                          ║
║                                                                              ║
║  arr(x1, y1, x2, y2, uid, colour)                                           ║
║      → straight arrow from (x1,y1) to (x2,y2)                              ║
║      → uid must match the figure number string, e.g. "1" inside fig1()     ║
║      → add  dash=True  for a dashed subClassOf arrow                        ║
║                                                                              ║
║  parr([(x1,y1), (x2,y2), (x3,y3)], uid, colour)                            ║
║      → L-shaped multi-segment path arrow (avoids routing through boxes)     ║
║      → add corners as extra waypoints, e.g. go right then down             ║
║                                                                              ║
║  lbl(x, y, "text", colour, anchor)                                          ║
║      → floating property label on an arrow                                  ║
║      → anchor = "middle" (default) | "start" | "end"                       ║
║                                                                              ║
║  COLOUR PALETTE  (edit C dict below to change any colour globally)          ║
║  ─────────────────────────────────────────────────────────────────           ║
║  teal   = KPI / Geography / Stakeholder classes                             ║
║  amber  = Archetype / Energy Load classes                                   ║
║  coral  = Scenario / Heat Density classes                                   ║
║  pink   = EV Mode / Tariff classes                                          ║
║  blue   = LV Network classes                                                ║
║  green  = Profile / Observation classes                                     ║
║  red    = Attribution / Overload classes                                    ║
║  purple = Visualisation / Report classes                                    ║
║  gray   = Data types, derived values, helper nodes                          ║
║                                                                              ║
║  ADDING A NEW DIAGRAM                                                        ║
║  ─────────────────────                                                       ║
║  1. Copy any existing fig function, e.g.  def fig9(): ...                   ║
║  2. Change uid to "9"                                                        ║
║  3. Add it to the  FIGS  dict at the bottom of this file                    ║
║  4. Re-run:  python3 gen_diagrams.py                                         ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import os
import sys
import webbrowser
from pathlib import Path

# ── Output path ───────────────────────────────────────────────────────────────
# Works in both a plain Python script AND a Jupyter / IPython notebook.
# In a notebook __file__ is not defined, so we fall back to the current
# working directory (the folder you opened in VS Code / JupyterLab).
try:
    HERE = Path(__file__).parent          # running as  python3 gen_diagrams.py
except NameError:
    HERE = Path.cwd()                     # running inside a Jupyter notebook

OUT_FILE = HERE / "ontology_diagrams.html"

# ── Open browser automatically after generation? ─────────────────────────────
# Change to False if you prefer to open manually.
AUTO_OPEN = True


# ═════════════════════════════════════════════════════════════════════════════
# DRAWING PRIMITIVES
# ─────────────────────────────────────────────────────────────────────────────
# All coordinates are in SVG user units (1 unit ≈ 1px at 96 DPI).
# Canvas width is always 900.  Height is set per-figure.
# ═════════════════════════════════════════════════════════════════════════════

STYLE = """<style>
  .th { font-family: Arial, sans-serif; font-size: 13px; font-weight: 700; fill: #1a1a1a }
  .ts { font-family: Arial, sans-serif; font-size: 11px; fill: #333 }
  .tl { font-family: Arial, sans-serif; font-size: 10px; fill: #555 }
</style>"""

def _defs(uid):
    """Arrow-head marker — uid keeps markers unique across figures on the same page."""
    return f"""<defs>
<marker id="ar{uid}" viewBox="0 0 10 10" refX="8" refY="5"
  markerWidth="6" markerHeight="6" orient="auto-start-reverse">
  <path d="M2 1L8 5L2 9" fill="none" stroke="context-stroke"
    stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
</marker>
</defs>"""

def svg(uid, W, H, body):
    """Wrap body in a complete SVG element with white background."""
    return (
        f'<svg id="svg{uid}" width="{W}" viewBox="0 0 {W} {H}" '
        f'xmlns="http://www.w3.org/2000/svg">'
        f'<rect width="{W}" height="{H}" fill="white"/>'
        + STYLE + _defs(uid) + body + '</svg>'
    )


# ── Colour palette ────────────────────────────────────────────────────────────
# _f = fill (light),  _s = stroke (medium),  _t = text (dark)
# Edit any value here and every diagram updates on next run.
C = {
    "teal_f":   "#E1F5EE",  "teal_s":   "#0F6E56",  "teal_t":   "#085041",
    "purple_f": "#EEEDFE",  "purple_s": "#534AB7",  "purple_t": "#26215C",
    "amber_f":  "#FAEEDA",  "amber_s":  "#854F0B",  "amber_t":  "#412402",
    "coral_f":  "#FAECE7",  "coral_s":  "#993C1D",  "coral_t":  "#4A1B0C",
    "pink_f":   "#FBEAF0",  "pink_s":   "#993556",  "pink_t":   "#4B1528",
    "blue_f":   "#E6F1FB",  "blue_s":   "#185FA5",  "blue_t":   "#042C53",
    "green_f":  "#EAF3DE",  "green_s":  "#3B6D11",  "green_t":  "#173404",
    "red_f":    "#FCEBEB",  "red_s":    "#A32D2D",  "red_t":    "#501313",
    "gray_f":   "#F1EFE8",  "gray_s":   "#5F5E5A",  "gray_t":   "#2C2C2A",
}


# ── Box ───────────────────────────────────────────────────────────────────────
def box(x, y, w, h, cf, cs, title, subtitle="", rx=6):
    """
    Coloured rectangle with centred title (and optional smaller subtitle).

    Parameters
    ----------
    x, y        : top-left corner
    w, h        : width and height
    cf, cs      : fill colour, stroke colour  (use C["teal_f"] etc.)
    title       : main label (bold, 13px)
    subtitle    : secondary label (regular, 11px) — leave "" to omit
    rx          : corner radius (default 6)

    Sizing guide
    ────────────
    Single-line box  →  h = 36
    Two-line box     →  h = 52
    Width rule       →  w = max(len(title)*9, len(subtitle)*7) + 32
    """
    r = (f'<rect x="{x}" y="{y}" width="{w}" height="{h}" rx="{rx}" '
         f'fill="{cf}" stroke="{cs}" stroke-width="0.8"/>')
    if subtitle:
        ty = y + h // 2 - 8
        sy = y + h // 2 + 8
        r += (f'<text class="th" x="{x+w//2}" y="{ty}" '
              f'text-anchor="middle" dominant-baseline="central" '
              f'fill="{C["gray_t"]}">{title}</text>')
        r += (f'<text class="ts" x="{x+w//2}" y="{sy}" '
              f'text-anchor="middle" dominant-baseline="central" '
              f'fill="{cs}">{subtitle}</text>')
    else:
        r += (f'<text class="th" x="{x+w//2}" y="{y+h//2}" '
              f'text-anchor="middle" dominant-baseline="central" '
              f'fill="{C["gray_t"]}">{title}</text>')
    return r


def gbox(x, y, w, h, title, subtitle="", rx=6):
    """Shorthand: grey data-type / helper box."""
    return box(x, y, w, h, C["gray_f"], C["gray_s"], title, subtitle, rx)


# ── Arrows ────────────────────────────────────────────────────────────────────
def arr(x1, y1, x2, y2, uid, col="#333", dash=False):
    """
    Straight arrow from (x1,y1) to (x2,y2).

    uid   : figure number string — must match the surrounding svg() call
    col   : stroke colour
    dash  : True for a dashed subClassOf arrow
    """
    d = ' stroke-dasharray="5 3"' if dash else ''
    return (f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" '
            f'stroke="{col}" stroke-width="0.9"{d} '
            f'marker-end="url(#ar{uid})"/>')


def parr(pts, uid, col="#333", dash=False):
    """
    Multi-segment path arrow.  Use this to route around boxes.

    pts  : list of (x, y) waypoints — the arrow ends at the last point
    uid  : figure number string
    col  : stroke colour
    dash : True for dashed line

    Example — go right then down:
        parr([(startX, startY), (cornerX, startY), (cornerX, endY)], uid, col)
    """
    d = " ".join(f"{'M' if i == 0 else 'L'}{x} {y}"
                 for i, (x, y) in enumerate(pts))
    da = ' stroke-dasharray="5 3"' if dash else ''
    return (f'<path d="{d}" fill="none" stroke="{col}" '
            f'stroke-width="0.9"{da} marker-end="url(#ar{uid})"/>')


# ── Labels ────────────────────────────────────────────────────────────────────
def lbl(x, y, text, col="#555", anchor="middle"):
    """
    Floating property label (10px grey).

    anchor : "middle" | "start" | "end"
    Place beside an arrow, never on top of a box.
    """
    return (f'<text class="tl" x="{x}" y="{y}" '
            f'text-anchor="{anchor}" fill="{col}">{text}</text>')


# ── Legend and caption ────────────────────────────────────────────────────────
def legend(y, items):
    """
    Horizontal colour legend at the bottom of a diagram.

    items : list of (fill_colour, stroke_colour, label_string)
    y     : vertical position of the legend row
    """
    x = 30
    r = ""
    for cf, cs, label in items:
        r += (f'<rect x="{x}" y="{y-9}" width="14" height="14" rx="3" '
              f'fill="{cf}" stroke="{cs}" stroke-width="0.8"/>')
        r += f'<text class="tl" x="{x+18}" y="{y+2}" fill="#555">{label}</text>'
        x += len(label) * 7 + 36
    return r


def figcap(W, H, text):
    """Centred figure caption at the very bottom of the SVG."""
    return (f'<text class="tl" x="{W//2}" y="{H-10}" '
            f'text-anchor="middle" fill="#999">{text}</text>')


# ═════════════════════════════════════════════════════════════════════════════
# FIG 0 — CONCEPTUAL MODEL (overview of all modules)
# ═════════════════════════════════════════════════════════════════════════════
def fig0():
    W, H = 900, 720
    u = "0"
    b = ""

    # ── Helper: dashed module boundary box with label ──────────────────────
    def modbox(x, y, w, h, label, col):
        return (
            f'<rect x="{x}" y="{y}" width="{w}" height="{h}" rx="10" '
            f'fill="none" stroke="{col}" stroke-width="0.8" stroke-dasharray="6 3"/>'
            f'<text class="tl" x="{x+8}" y="{y+14}" fill="{col}">{label}</text>'
        )

    # ── Module boundary boxes (top row) ────────────────────────────────────
    b += modbox(20,  20, 180, 160, "KPI",         C["teal_s"])
    b += modbox(210, 20, 200, 160, "Archetype",   C["amber_s"])
    b += modbox(420, 20, 220, 160, "LV Network",  C["blue_s"])
    b += modbox(650, 20, 230, 160, "Geography",   C["teal_s"])

    # ── Module boundary boxes (middle row) ─────────────────────────────────
    b += modbox(20,  200, 180, 200, "Scenario",      C["coral_s"])
    b += modbox(210, 200, 200, 200, "Energy Load",   C["amber_s"])
    b += modbox(420, 200, 220, 200, "Attribution",   C["red_s"])
    b += modbox(650, 200, 230, 200, "Heat Density",  C["coral_s"])

    # ── Module boundary boxes (bottom row) ─────────────────────────────────
    b += modbox(20,  420, 400, 140, "Profile / Observation",  C["green_s"])
    b += modbox(430, 420, 450, 140, "Visualisation / Report", C["purple_s"])

    # ── Class nodes — KPI module ───────────────────────────────────────────
    b += box(30,  44,  160, 36, C["teal_f"],  C["teal_s"],  "nedt:KPI")
    b += box(30, 100,  160, 36, C["teal_f"],  C["teal_s"],  "nedt:Stakeholder")
    b += box(30, 152,  160, 36, C["teal_f"],  C["teal_s"],  "nedt:PerformanceGoal")

    # ── Class nodes — Archetype module ────────────────────────────────────
    b += box(220,  44, 180, 36, C["amber_f"], C["amber_s"], "nedt:Archetype")
    b += box(220, 100, 180, 36, C["amber_f"], C["amber_s"], "nedt:ArchetypeDescriptor")
    b += box(220, 152, 180, 36, C["amber_f"], C["amber_s"], "nedt:ArchetypeCount")

    # ── Class nodes — LV Network module ───────────────────────────────────
    b += box(430,  44, 200, 36, C["blue_f"],  C["blue_s"],  "nedt:LVStation")
    b += box(430, 100, 200, 36, C["blue_f"],  C["blue_s"],  "nedt:HullPolygon")
    b += box(430, 152, 200, 36, C["blue_f"],  C["blue_s"],  "nedt:StationCapacity")

    # ── Class nodes — Geography module ────────────────────────────────────
    b += box(660,  44, 210, 36, C["teal_f"],  C["teal_s"],  "nedt:County")
    b += box(660, 100, 210, 36, C["teal_f"],  C["teal_s"],  "nedt:GeoLocation")
    b += box(660, 152, 210, 36, C["teal_f"],  C["teal_s"],  "nedt:BuildingShare")

    # ── Class nodes — Scenario module ─────────────────────────────────────
    b += box(30, 224, 160, 36, C["coral_f"], C["coral_s"], "nedt:HPScenario")
    b += box(30, 280, 160, 36, C["coral_f"], C["coral_s"], "nedt:EVScenario")
    b += box(30, 336, 160, 36, C["coral_f"], C["coral_s"], "nedt:TransitionLedger")

    # ── Class nodes — Energy Load module ──────────────────────────────────
    b += box(220, 224, 180, 36, C["amber_f"], C["amber_s"], "nedt:ElectricLoad")
    b += box(220, 280, 180, 36, C["amber_f"], C["amber_s"], "nedt:HeatLoad")
    b += box(220, 336, 180, 36, C["gray_f"],  C["gray_s"],  "nedt:CO2Emission")

    # ── Class nodes — Attribution module ──────────────────────────────────
    b += box(430, 224, 200, 36, C["red_f"],   C["red_s"],   "nedt:OverloadedStation")
    b += box(430, 280, 200, 36, C["red_f"],   C["red_s"],   "nedt:OverloadAttribution")
    b += box(430, 336, 200, 36, C["red_f"],   C["red_s"],   "nedt:RiskTable")

    # ── Class nodes — Heat Density module ─────────────────────────────────
    b += box(660, 224, 210, 36, C["coral_f"], C["coral_s"], "nedt:HeatDensityGDF")
    b += box(660, 280, 210, 36, C["coral_f"], C["coral_s"], "nedt:HeatFuelComponent")
    b += box(660, 336, 210, 36, C["coral_f"], C["coral_s"], "nedt:HeatDensity")

    # ── Class nodes — Profile module ──────────────────────────────────────
    b += box(30,  444, 180, 36, C["green_f"],  C["green_s"],  "nedt:HourlyProfile")
    b += box(230, 444, 180, 36, C["green_f"],  C["green_s"],  "nedt:NexsysProfile")
    b += box(30,  500, 380, 36, C["gray_f"],   C["gray_s"],   "ssn:ObservationValue")

    # ── Class nodes — Visualisation module ────────────────────────────────
    b += box(440, 444, 200, 36, C["purple_f"], C["purple_s"], "nedt:FigCollector")
    b += box(440, 500, 200, 36, C["purple_f"], C["purple_s"], "nedt:DeckGLMap")
    b += box(650, 444, 220, 36, C["purple_f"], C["purple_s"], "nedt:ChartJSChart")
    b += box(650, 500, 220, 36, C["gray_f"],   C["gray_s"],   "nedt:HTMLReport")

    # ── Inter-module arrows ────────────────────────────────────────────────
    ac = C["teal_s"]

    # Within KPI module
    b += arr(110, 136, 110, 152, u, ac)
    b += arr(110,  80, 110, 100, u, ac)

    # KPI → Archetype (evaluates)
    b += arr(190, 62, 220, 62, u, "#888")
    b += lbl(205, 56, "evaluates", "#888")

    # Archetype → ElectricLoad (computesLoad)
    b += parr([(310, 188), (310, 224)], u, C["amber_s"])
    b += lbl(316, 210, "computesLoad", C["amber_s"], "start")

    # Archetype → HourlyProfile (usesProfile)
    b += parr([(310, 188), (310, 410), (120, 410), (120, 444)], u, C["green_s"])
    b += lbl(116, 400, "usesProfile", C["green_s"], "end")

    # Scenario → Archetype (modifies)
    b += parr([(190, 242), (210, 242)], u, C["coral_s"])
    b += lbl(196, 236, "modifies", C["coral_s"], "start")

    # ElectricLoad → LVStation (allocatedTo)
    b += parr([(400, 242), (430, 242)], u, C["amber_s"])
    b += lbl(406, 236, "allocatedTo", C["amber_s"], "start")

    # LVStation → Attribution (triggers overload analysis)
    b += arr(630, 242, 430, 280, u, C["blue_s"])
    b += lbl(540, 268, "triggers", C["blue_s"])

    # County → LVStation (contains)
    b += parr([(660, 62), (630, 62)], u, C["teal_s"])
    b += lbl(648, 56, "contains", C["teal_s"], "end")

    # County → Archetype (hasArchetype) — routed below all modules
    b += parr([(660, 100), (660, 390), (200, 390), (200, 152)], u, C["teal_s"])
    b += lbl(430, 388, "hasArchetype", C["teal_s"])

    # HeatLoad → HeatDensity (→ density computation)
    b += parr([(400, 298), (660, 298)], u, C["amber_s"])
    b += lbl(530, 292, "→ density", C["amber_s"])

    # Profile → ObservationValue
    b += arr(120, 480, 120, 500, u, C["green_s"])

    # FigCollector → DeckGLMap
    b += arr(540, 480, 540, 500, u, C["purple_s"])

    # FigCollector → ChartJSChart
    b += arr(640, 462, 650, 462, u, C["purple_s"])

    # ── Legend ────────────────────────────────────────────────────────────
    b += (f'<line x1="30" y1="600" x2="60" y2="600" stroke="{ac}" '
          f'stroke-width="0.9" marker-end="url(#ar{u})"/>')
    b += f'<text class="tl" x="66" y="604" fill="{ac}">object property</text>'
    b += (f'<line x1="200" y1="600" x2="230" y2="600" stroke="#888" '
          f'stroke-width="0.9" stroke-dasharray="5 3"/>')
    b += f'<text class="tl" x="236" y="604" fill="#888">module boundary (dashed)</text>'

    b += legend(638, [
        (C["teal_f"],   C["teal_s"],   "KPI / County"),
        (C["amber_f"],  C["amber_s"],  "Archetype / Load"),
        (C["blue_f"],   C["blue_s"],   "LV Network"),
        (C["coral_f"],  C["coral_s"],  "Scenario / Heat"),
        (C["red_f"],    C["red_s"],    "Attribution"),
        (C["purple_f"], C["purple_s"], "Visualisation"),
        (C["green_f"],  C["green_s"],  "Profile"),
    ])
    b += figcap(W, H, "Fig. 0 — Conceptual model overview  |  National Energy Digital Twin (NEDT) Ontology")
    return svg(u, W, H, b)


# ═════════════════════════════════════════════════════════════════════════════
# FIG 1 — KPI MODULE
# ═════════════════════════════════════════════════════════════════════════════
def fig1():
    W, H = 900, 580
    u = "1"
    b = ""
    ac = C["teal_s"]

    # ── Stakeholder → PerformanceGoal → KPI (top-centre column) ───────────
    b += box(290, 30, 220, 36, C["teal_f"], C["teal_s"], "nedt:Stakeholder")
    b += arr(400, 66, 400, 94, u, ac)
    b += lbl(406, 82, "hasPerformanceGoal", ac, "start")

    b += box(270, 94, 260, 36, C["teal_f"], C["teal_s"], "nedt:PerformanceGoal")
    b += arr(400, 130, 400, 158, u, ac)
    b += lbl(406, 146, "hasAssociatedKPI", ac, "start")

    b += box(310, 158, 180, 36, C["purple_f"], C["purple_s"], "nedt:KPI")

    # Identifier and definition data properties (pointing left)
    b += parr([(310, 168), (240, 168)], u, "#888")
    b += lbl(270, 162, "dct:identifier", "#888")
    b += gbox(130, 156, 104, 28, "xsd:string")

    b += parr([(310, 180), (240, 180)], u, "#888")
    b += lbl(260, 192, "nedt:hasKPIDefinition", "#888")
    b += gbox(130, 170, 104, 28, "xsd:string")

    # ── KPI subclasses (right branch) ─────────────────────────────────────
    b += parr([(490, 176), (560, 176), (560, 236)], u, "#888", dash=True)
    b += lbl(562, 210, "subClassOf", "#888", "start")
    b += parr([(560, 236), (560, 250)], u, "#888", dash=True)
    b += parr([(560, 280), (560, 294)], u, "#888", dash=True)
    b += parr([(560, 324), (560, 338)], u, "#888", dash=True)
    b += box(590, 250, 200, 36, C["purple_f"], C["purple_s"], "nedt:StrategicKPI")
    b += box(590, 294, 200, 36, C["purple_f"], C["purple_s"], "nedt:TacticalKPI")
    b += box(590, 338, 200, 36, C["purple_f"], C["purple_s"], "nedt:OperationalKPI")
    b += arr(690, 286, 690, 294, u, "#bbb", dash=True)
    b += arr(690, 330, 690, 338, u, "#bbb", dash=True)
    b += lbl(696, 292, "hasDisaggregation", "#bbb", "start")

    # ── KPIEvaluatedObject (left branch) ──────────────────────────────────
    b += parr([(310, 194), (120, 194), (120, 250)], u, ac)
    b += lbl(110, 230, "hasKPIEvaluatedObject", ac, "end")
    b += box(30, 250, 280, 36, C["teal_f"], C["teal_s"], "nedt:KPIEvaluatedObject")
    b += parr([(100, 286),  (80, 310)], u, "#888", dash=True)
    b += parr([(170, 286), (170, 310)], u, "#888", dash=True)
    b += parr([(240, 286), (260, 310)], u, "#888", dash=True)
    b += box( 30, 310, 100, 30, C["teal_f"], C["teal_s"], "nedt:County")
    b += box(136, 310, 100, 30, C["teal_f"], C["teal_s"], "nedt:LVStation")
    b += box(242, 310, 104, 30, C["teal_f"], C["teal_s"], "nedt:Archetype")

    # ── KPICalculation (centre column, below KPI) ─────────────────────────
    b += arr(400, 194, 400, 258, u, ac)
    b += lbl(406, 230, "hasCalculation", ac, "start")
    b += box(300, 258, 200, 36, C["purple_f"], C["purple_s"], "nedt:KPICalculation")

    # ssn:hasInput (left)
    b += parr([(300, 276), (200, 276), (200, 350)], u, ac)
    b += lbl(196, 316, "ssn:hasInput", ac, "end")

    # ssn:hasOutput (right)
    b += parr([(500, 276), (540, 276), (540, 350)], u, ac)
    b += lbl(546, 316, "ssn:hasOutput", ac, "start")

    # hasEvalTimeStep (straight down)
    b += arr(400, 294, 400, 350, u, ac)
    b += lbl(406, 326, "hasEvalTimeStep", ac, "start")

    # ── DatumSource (left column) ──────────────────────────────────────────
    b += box(130, 350, 200, 36, C["gray_f"], C["gray_s"], "nedt:DatumSource")
    b += arr(230, 386, 230, 412, u, ac)
    b += lbl(236, 402, "isProvidedBy", ac, "start")
    b += box(130, 412, 200, 36, C["gray_f"], C["gray_s"], "ssn:Observation")

    # ── time:Interval (centre column) ─────────────────────────────────────
    b += box(310, 350, 180, 36, C["gray_f"], C["gray_s"], "time:Interval")
    b += parr([(350, 386), (330, 412)], u, "#888", dash=True)
    b += parr([(430, 386), (450, 412)], u, "#888", dash=True)
    b += box(284, 412, 100, 30, C["gray_f"], C["gray_s"], "time:hasBegin")
    b += box(390, 412, 100, 30, C["gray_f"], C["gray_s"], "time:hasEnd")
    b += arr(334, 442, 334, 462, u, "#888", dash=True)
    b += arr(440, 442, 440, 462, u, "#888", dash=True)
    b += gbox(286, 462, 100, 28, "time:Instant")
    b += gbox(392, 462, 110, 28, "xsd:dateTime")

    # ── KPIValue (right column) ───────────────────────────────────────────
    b += box(492, 350, 170, 36, C["purple_f"], C["purple_s"], "nedt:KPIValue")
    b += arr(577, 386, 577, 412, u, "#888")
    b += lbl(583, 402, "hasUnit", "#888", "start")
    b += gbox(478, 412, 198, 30, "om:Unit_of_measure")
    b += arr(577, 442, 577, 462, u, "#888")
    b += gbox(516, 462, 122, 28, "xsd:decimal")

    # ── Legend ────────────────────────────────────────────────────────────
    b += (f'<line x1="30" y1="528" x2="60" y2="528" stroke="{ac}" '
          f'stroke-width="0.9" marker-end="url(#ar{u})"/>')
    b += f'<text class="tl" x="66" y="532" fill="{ac}">object property</text>'
    b += (f'<line x1="200" y1="528" x2="230" y2="528" stroke="#888" '
          f'stroke-width="0.9" stroke-dasharray="5 3" marker-end="url(#ar{u})"/>')
    b += f'<text class="tl" x="236" y="532" fill="#888">subClassOf</text>'
    b += legend(550, [
        (C["purple_f"], C["purple_s"], "KPI classes"),
        (C["teal_f"],   C["teal_s"],   "domain objects"),
        (C["gray_f"],   C["gray_s"],   "data types"),
    ])
    b += figcap(W, H, "Fig. 1 — KPI module  |  National Energy Digital Twin (NEDT) Ontology")
    return svg(u, W, H, b)


# ═════════════════════════════════════════════════════════════════════════════
# FIG 2 — ARCHETYPE MODULE
# ═════════════════════════════════════════════════════════════════════════════
def fig2():
    W, H = 900, 540
    u = "2"
    b = ""
    ac = C["amber_s"]

    b += box(340, 30, 220, 36, C["amber_f"], C["amber_s"], "nedt:Archetype")

    # ── Left branch: hasDescriptor ────────────────────────────────────────
    b += parr([(340, 48), (160, 48), (160, 80)], u, ac)
    b += lbl(156, 64, "hasDescriptor", ac, "end")
    b += box(40, 80, 240, 36, C["amber_f"], C["amber_s"], "nedt:ArchetypeDescriptor")

    # 5 descriptor subclasses — 84px wide, 10px gap, starting at x=30
    for i, label in enumerate(["BuildType", "BER", "Occupancy", "HeatSystem", "WxClass"]):
        fx = 72 + i * 94
        b += parr([(fx, 116), (fx, 140)], u, "#888", dash=True)
        b += box(30 + i * 94, 140, 84, 30, C["amber_f"], C["amber_s"], label)

    # BER canonical mapping
    b += arr(160, 170, 130, 200, u, "#888")
    b += lbl(126, 196, "map_ber_letter", "#888", "end")
    b += gbox(30, 200, 140, 28, "A|B|C|D|E|F|G")

    # HeatSystem canonical mapping
    b += arr(342, 170, 342, 200, u, "#888")
    b += lbl(348, 196, "map_heat", "#888", "start")
    b += box(270, 200, 180, 40, C["gray_f"], C["gray_s"],
             "oilboiler | gasboiler", "heatpump")

    # ── Right branch: hasCount ────────────────────────────────────────────
    b += parr([(560, 48), (720, 48), (720, 80)], u, ac)
    b += lbl(716, 64, "hasCount", ac, "end")
    b += box(590, 80, 250, 36, C["amber_f"], C["amber_s"], "nedt:ArchetypeCount")
    for i, (lname, sub) in enumerate([("AC_ORIG","total"), ("AC_EV","with EV"), ("AC_NOEV","no EV")]):
        fx = 644 + i * 72
        b += parr([(fx, 116), (fx, 140)], u, "#888", dash=True)
        b += box(598 + i * 70, 140, 68, 44, C["amber_f"], C["amber_s"], lname, sub)
    b += lbl(715, 200, "AC_NOEV = AC_ORIG − AC_EV  (clipped ≥ 0)", "#888")

    # ── Centre: ProfileKey ────────────────────────────────────────────────
    b += arr(450, 66, 450, 80, u, ac)
    b += lbl(456, 74, "usesProfile", ac, "start")
    b += box(310, 80, 280, 52, C["green_f"], C["green_s"],
             "nedt:ProfileKey", "{Build}_{BER}_{Occ}_{Heat}_{Wx}.csv")

    # 3 profile subclasses
    b += parr([(380, 132), (340, 152)], u, "#888", dash=True)
    b += parr([(450, 132), (450, 152)], u, "#888", dash=True)
    b += parr([(530, 132), (570, 152)], u, "#888", dash=True)
    b += box(284, 152, 120, 44, C["green_f"], C["green_s"], "NoEV profile", "ELEC_DIR")
    b += box(410, 152, 120, 44, C["green_f"], C["green_s"], "EV profile",   "ELEC_DIR")
    b += box(536, 152, 120, 44, C["green_f"], C["green_s"], "Heat profile", "HEAT_DIR")

    # BERFallbackRule
    b += arr(450, 196, 450, 218, u, C["green_s"])
    b += lbl(456, 210, "hasBERFallback", C["green_s"], "start")
    b += box(290, 218, 320, 36, C["gray_f"], C["gray_s"],
             "nedt:BERFallbackRule", "A→B→C  |  B→C")

    # ObservationValue
    b += arr(450, 254, 450, 276, u, C["green_s"])
    b += lbl(456, 268, "hasObservation", C["green_s"], "start")
    b += box(290, 276, 320, 48, C["gray_f"], C["gray_s"],
             "ssn:ObservationValue", "8 760 hourly GWh/h values")

    # TransitionKey
    b += arr(450, 324, 450, 348, u, ac)
    b += lbl(456, 338, "hasTransitionKey", ac, "start")
    b += box(240, 348, 420, 48, C["coral_f"], C["coral_s"],
             "nedt:TransitionKey", "Build|BER|Occ|HeatSys|WxClass|EVFlag")

    # ── Legend ────────────────────────────────────────────────────────────
    b += (f'<line x1="30" y1="456" x2="60" y2="456" stroke="{ac}" '
          f'stroke-width="0.9" marker-end="url(#ar{u})"/>')
    b += f'<text class="tl" x="66" y="460" fill="{ac}">object property</text>'
    b += (f'<line x1="200" y1="456" x2="230" y2="456" stroke="#888" '
          f'stroke-width="0.9" stroke-dasharray="5 3" marker-end="url(#ar{u})"/>')
    b += f'<text class="tl" x="236" y="460" fill="#888">subClassOf</text>'
    b += legend(478, [
        (C["amber_f"],  C["amber_s"],  "archetype"),
        (C["green_f"],  C["green_s"],  "profile"),
        (C["coral_f"],  C["coral_s"],  "transition"),
        (C["gray_f"],   C["gray_s"],   "data type"),
    ])
    b += figcap(W, H, "Fig. 2 — Archetype module  |  National Energy Digital Twin (NEDT) Ontology")
    return svg(u, W, H, b)


# ═════════════════════════════════════════════════════════════════════════════
# FIG 3 — SCENARIO MODULE
# ═════════════════════════════════════════════════════════════════════════════
def fig3():
    W, H = 900, 600
    u = "3"
    b = ""
    ac = C["coral_s"]

    b += box(330, 30, 240, 36, C["coral_f"], C["coral_s"], "nedt:Scenario")

    # ── Left: HPScenario ──────────────────────────────────────────────────
    b += parr([(330, 48), (120, 48), (120, 80)], u, ac)
    b += lbl(116, 64, "hasHP", ac, "end")
    b += box(30, 80, 180, 36, C["coral_f"], C["coral_s"], "nedt:HPScenario")
    b += arr(120, 116, 120, 140, u, "#888")
    b += box(30, 140, 180, 44, C["gray_f"], C["gray_s"],
             "oil_pct: xsd:float", "gas_pct: xsd:float")
    b += arr(50, 184, 50, 208, u, "#888")
    b += lbl(46, 202, "eligibleBER", "#888", "end")
    b += gbox(30, 208, 180, 28, "BER  C | D | E  →  B")
    b += arr(120, 236, 120, 260, u, ac)
    b += lbl(126, 252, "produces", ac, "start")
    b += box(30, 260, 180, 52, C["coral_f"], C["coral_s"],
             "nedt:HPTransition", "from_key | to_key")
    b += arr(120, 312, 120, 336, u, "#888")
    b += gbox(30, 336, 180, 28, "xsd:integer  (LRA)")

    # ── Centre: TransitionLedger ───────────────────────────────────────────
    b += arr(450, 66, 450, 80, u, ac)
    b += lbl(456, 74, "recordsIn", ac, "start")
    b += box(280, 80, 340, 52, C["coral_f"], C["coral_s"],
             "nedt:TransitionLedger", "County | Reason | UnitsMoved")
    b += parr([(360, 132), (320, 156)], u, "#888", dash=True)
    b += parr([(450, 132), (450, 156)], u, "#888", dash=True)
    b += parr([(540, 132), (580, 156)], u, "#888", dash=True)
    b += box(262, 156, 114, 30, C["coral_f"], C["coral_s"], "HP transition")
    b += box(382, 156, 136, 30, C["coral_f"], C["coral_s"], "EV increase")
    b += box(524, 156, 120, 30, C["coral_f"], C["coral_s"], "EV mode/tariff")

    # LargestRemainderAlloc
    b += arr(450, 186, 450, 208, u, ac)
    b += lbl(456, 200, "allocatedBy", ac, "start")
    b += box(262, 208, 376, 52, C["gray_f"], C["gray_s"],
             "nedt:LargestRemainderAlloc",
             "weights | total → exact integer sum")

    # ── Right: EVScenario ─────────────────────────────────────────────────
    b += parr([(570, 48), (760, 48), (760, 80)], u, ac)
    b += lbl(666, 42, "hasEV", ac)
    b += box(680, 80, 190, 36, C["pink_f"], C["pink_s"], "nedt:EVScenario")
    b += arr(775, 116, 775, 140, u, "#888")
    b += gbox(680, 140, 190, 28, "ev_increase_pct: float")
    b += arr(775, 168, 775, 192, u, ac)
    b += lbl(781, 182, "hasMode", ac, "start")
    b += box(680, 192, 190, 36, C["pink_f"], C["pink_s"], "nedt:EVMode")
    b += parr([(730, 228), (700, 252)], u, "#888", dash=True)
    b += parr([(775, 228), (775, 252)], u, "#888", dash=True)
    b += parr([(820, 228), (850, 252)], u, "#888", dash=True)
    b += box(654, 252,  98, 28, C["pink_f"], C["pink_s"], "Uncontrolled")
    b += box(756, 252,  56, 28, C["pink_f"], C["pink_s"], "V1H")
    b += box(820, 252,  60, 28, C["pink_f"], C["pink_s"], "V2H")

    # Tariff
    b += arr(775, 280, 775, 304, u, ac)
    b += lbl(781, 295, "hasTariff", ac, "start")
    b += box(680, 304, 190, 36, C["pink_f"], C["pink_s"], "nedt:Tariff")
    b += parr([(730, 340), (700, 364)], u, "#888", dash=True)
    b += parr([(775, 340), (775, 364)], u, "#888", dash=True)
    b += parr([(820, 340), (850, 364)], u, "#888", dash=True)
    b += box(660, 364, 80, 28, C["pink_f"], C["pink_s"], "Tier3")
    b += box(748, 364, 56, 28, C["pink_f"], C["pink_s"], "Tier4")
    b += box(816, 364, 68, 28, C["pink_f"], C["pink_s"], "Dynamic")

    # NexsysProfile
    b += arr(775, 392, 775, 416, u, ac)
    b += lbl(781, 408, "usesNexsys", ac, "start")
    b += box(660, 416, 200, 44, C["green_f"], C["green_s"],
             "nedt:NexsysProfile", "EV Charging power col")

    # ── Legend ────────────────────────────────────────────────────────────
    b += (f'<line x1="30" y1="518" x2="60" y2="518" stroke="{ac}" '
          f'stroke-width="0.9" marker-end="url(#ar{u})"/>')
    b += f'<text class="tl" x="66" y="522" fill="{ac}">object property</text>'
    b += (f'<line x1="200" y1="518" x2="230" y2="518" stroke="#888" '
          f'stroke-width="0.9" stroke-dasharray="5 3" marker-end="url(#ar{u})"/>')
    b += f'<text class="tl" x="236" y="522" fill="#888">subClassOf</text>'
    b += legend(542, [
        (C["coral_f"],  C["coral_s"],  "scenario"),
        (C["pink_f"],   C["pink_s"],   "EV mode"),
        (C["green_f"],  C["green_s"],  "Nexsys profile"),
        (C["gray_f"],   C["gray_s"],   "data type"),
    ])
    b += figcap(W, H, "Fig. 3 — Scenario module  |  National Energy Digital Twin (NEDT) Ontology")
    return svg(u, W, H, b)


# ═════════════════════════════════════════════════════════════════════════════
# FIG 4 — ENERGY LOAD MODULE
# ═════════════════════════════════════════════════════════════════════════════
def fig4():
    W, H = 900, 520
    u = "4"
    b = ""
    ac = C["amber_s"]

    b += box(330, 30, 240, 36, C["amber_f"], C["amber_s"], "nedt:EnergyLoad")

    # ── Left: ElectricLoad ────────────────────────────────────────────────
    b += parr([(330, 48), (140, 48), (140, 80)], u, ac)
    b += lbl(136, 64, "hasElec", ac, "end")
    b += box(40, 80, 200, 36, C["amber_f"], C["amber_s"], "nedt:ElectricLoad")
    b += parr([(100, 116), ( 80, 140)], u, "#888", dash=True)
    b += parr([(180, 116), (200, 140)], u, "#888", dash=True)
    b += box( 20, 140, 116, 44, C["amber_f"], C["amber_s"],
              "NoEV demand", "profile × AC_NOEV")
    b += box(142, 140, 148, 44, C["amber_f"], C["amber_s"],
              "EV demand", "NoEV base + charging")
    b += parr([(176, 184), (160, 208)], u, "#888", dash=True)
    b += parr([(236, 184), (260, 208)], u, "#888", dash=True)
    b += box( 92, 208, 128, 44, C["gray_f"],  C["gray_s"],
              "Standard EV", "ELEC_DIR full")
    b += box(226, 208, 148, 44, C["pink_f"],  C["pink_s"],
              "Mode-assigned", "Nexsys charging")
    b += arr(140, 252, 140, 276, u, ac)
    b += lbl(146, 266, "produces", ac, "start")
    b += box( 40, 276, 200, 44, C["gray_f"],  C["gray_s"],
              "nedt:LoadMetric", "annual_gwh | peak_kw")

    # ── Centre: LoadCalculation ───────────────────────────────────────────
    b += arr(450, 66, 450, 80, u, ac)
    b += lbl(456, 74, "computedBy", ac, "start")
    b += box(290, 80, 320, 52, C["gray_f"], C["gray_s"],
             "nedt:LoadCalculation",
             "elec_base | elec_scn | heat_base | heat_scn")
    b += parr([(370, 132), (320, 156)], u, "#888", dash=True)
    b += parr([(530, 132), (580, 156)], u, "#888", dash=True)
    b += box(260, 156, 120, 30, C["amber_f"], C["amber_s"], "Baseline loads")
    b += box(534, 156, 120, 30, C["coral_f"], C["coral_s"], "Scenario loads")
    b += lbl(450, 170, "delta = scn − base", "#888")
    b += arr(450, 186, 450, 208, u, "#888")
    b += box(290, 208, 320, 28, C["gray_f"], C["gray_s"],
             "miss_e_base | miss_h_base  (skipped)")

    # ── Right: HeatLoad ───────────────────────────────────────────────────
    b += parr([(570, 48), (760, 48), (760, 80)], u, ac)
    b += lbl(666, 42, "hasHeat", ac)
    b += box(680, 80, 190, 36, C["coral_f"], C["coral_s"], "nedt:HeatLoad")
    b += parr([(730, 116), (700, 140)], u, "#888", dash=True)
    b += parr([(775, 116), (775, 140)], u, "#888", dash=True)
    b += parr([(820, 116), (850, 140)], u, "#888", dash=True)
    b += box(656, 140,  90, 44, C["coral_f"], C["coral_s"], "OilHeat",  "oilboiler")
    b += box(752, 140,  78, 44, C["coral_f"], C["coral_s"], "GasHeat",  "gasboiler")
    b += box(836, 140,  56, 44, C["coral_f"], C["coral_s"], "HP",       "heatpump")
    b += parr([(858, 184), (858, 208)], u, "#888")
    b += lbl(864, 198, "usesNoEVProfile", "#888", "start")
    b += gbox(680, 208, 188, 28, "non-EV filename only")
    b += arr(775, 184, 775, 236, u, ac)
    b += lbl(781, 218, "hasCO2", ac, "start")
    b += box(680, 236, 190, 64, C["gray_f"], C["gray_s"],
             "nedt:CO2Emission", "elec 0.226 | gas 0.184 | oil 0.274")
    b += lbl(726, 320, "(kgCO₂/kWh)  |  HP direct = 0", "#888")

    # ── Legend ────────────────────────────────────────────────────────────
    b += (f'<line x1="30" y1="440" x2="60" y2="440" stroke="{ac}" '
          f'stroke-width="0.9" marker-end="url(#ar{u})"/>')
    b += f'<text class="tl" x="66" y="444" fill="{ac}">object property</text>'
    b += (f'<line x1="200" y1="440" x2="230" y2="440" stroke="#888" '
          f'stroke-width="0.9" stroke-dasharray="5 3" marker-end="url(#ar{u})"/>')
    b += f'<text class="tl" x="236" y="444" fill="#888">subClassOf</text>'
    b += legend(464, [
        (C["amber_f"], C["amber_s"], "electricity"),
        (C["coral_f"], C["coral_s"], "heat"),
        (C["pink_f"],  C["pink_s"],  "EV mode"),
        (C["gray_f"],  C["gray_s"],  "data type"),
    ])
    b += figcap(W, H, "Fig. 4 — Energy load module  |  National Energy Digital Twin (NEDT) Ontology")
    return svg(u, W, H, b)


# ═════════════════════════════════════════════════════════════════════════════
# FIG 5 — LV NETWORK MODULE
# ═════════════════════════════════════════════════════════════════════════════
def fig5():
    W, H = 900, 560
    u = "5"
    b = ""
    ac = C["blue_s"]

    b += box(330, 30, 240, 36, C["blue_f"], C["blue_s"], "nedt:LVStation")

    # ── Left: HullPolygon ─────────────────────────────────────────────────
    b += parr([(330, 48), (140, 48), (140, 80)], u, ac)
    b += lbl(136, 64, "hasHull", ac, "end")
    b += box(40, 80, 200, 36, C["blue_f"], C["blue_s"], "nedt:HullPolygon")
    b += parr([(100, 116), ( 80, 140)], u, "#888", dash=True)
    b += parr([(180, 116), (200, 140)], u, "#888", dash=True)
    b += gbox(28,  140, 100, 28, "geo:Polygon")
    b += box(134, 140, 148, 44, C["gray_f"], C["gray_s"],
             "area_km2", "EPSG:2157 (km²)")

    # ── Centre: StationCapacity ───────────────────────────────────────────
    b += arr(450, 66, 450, 80, u, ac)
    b += lbl(456, 74, "hasCapacity", ac, "start")
    b += box(290, 80, 320, 36, C["blue_f"], C["blue_s"], "nedt:StationCapacity")
    b += parr([(380, 116), (340, 140)], u, "#888", dash=True)
    b += parr([(520, 116), (560, 140)], u, "#888", dash=True)
    b += gbox(270, 140, 140, 28, "designed_kva: float")
    b += gbox(420, 140, 176, 28, "default: 200 kVA")

    # StationLoad
    b += arr(450, 168, 450, 192, u, ac)
    b += lbl(456, 183, "hasLoad", ac, "start")
    b += box(270, 192, 360, 52, C["blue_f"], C["blue_s"],
             "nedt:StationLoad", "annual_gwh | peak_kw | n_bldg | share")

    # Utilisation
    b += arr(450, 244, 450, 268, u, ac)
    b += lbl(456, 259, "hasUtilisation", ac, "start")
    b += box(290, 268, 320, 44, C["gray_f"], C["gray_s"],
             "nedt:Utilisation", "peak_kw / designed_kva")

    # UtilisationStatus
    b += arr(450, 312, 450, 336, u, ac)
    b += lbl(456, 327, "hasStatus", ac, "start")
    b += box(290, 336, 320, 36, C["blue_f"], C["blue_s"], "nedt:UtilisationStatus")

    # 5 status bands — 84px wide, 14px gap, 5 boxes total width = 504 → start x=54
    band_data = [
        ("Green",  "≤90%",      C["green_f"], C["green_s"]),
        ("Green",  "90–95%",    C["green_f"], C["green_s"]),
        ("Yellow", "95–99%",    C["amber_f"], C["amber_s"]),
        ("Maroon", "99–100%",   C["red_f"],   C["red_s"]),
        ("Red",    ">100%",     C["red_f"],   C["red_s"]),
    ]
    bw, gap, sx = 84, 14, 54
    for i, (bl, bs, cf, cs) in enumerate(band_data):
        fx = sx + i * (bw + gap) + bw // 2
        b += parr([(fx, 372), (fx, 396)], u, "#888", dash=True)
        b += box(sx + i * (bw + gap), 396, bw, 44, cf, cs, bl, bs)

    # ── Right: BuildingShare ──────────────────────────────────────────────
    b += parr([(570, 48), (760, 48), (760, 80)], u, ac)
    b += lbl(666, 42, "hasBldgShare", ac)
    b += box(680,  80, 190, 36, C["blue_f"], C["blue_s"], "nedt:BuildingShare")
    b += arr(775, 116, 775, 140, u, "#888")
    b += box(680, 140, 190, 44, C["gray_f"], C["gray_s"],
             "n_bldg: integer", "share: float")
    b += arr(775, 184, 775, 208, u, ac)
    b += lbl(781, 198, "derivedFrom", ac, "start")
    b += gbox(680, 208, 190, 44, "closest_building", "_station.csv")
    b += arr(775, 252, 775, 276, u, ac)
    b += lbl(781, 266, "inCounty", ac, "start")
    b += gbox(680, 276, 190, 28, "xsd:boolean (map flag)")

    # ── Legend ────────────────────────────────────────────────────────────
    b += (f'<line x1="30" y1="466" x2="60" y2="466" stroke="{ac}" '
          f'stroke-width="0.9" marker-end="url(#ar{u})"/>')
    b += f'<text class="tl" x="66" y="470" fill="{ac}">object property</text>'
    b += (f'<line x1="200" y1="466" x2="230" y2="466" stroke="#888" '
          f'stroke-width="0.9" stroke-dasharray="5 3" marker-end="url(#ar{u})"/>')
    b += f'<text class="tl" x="236" y="470" fill="#888">subClassOf</text>'
    b += legend(490, [
        (C["blue_f"],  C["blue_s"],  "LV classes"),
        (C["green_f"], C["green_s"], "green band"),
        (C["amber_f"], C["amber_s"], "yellow band"),
        (C["red_f"],   C["red_s"],   "red/maroon band"),
        (C["gray_f"],  C["gray_s"],  "data type"),
    ])
    b += figcap(W, H, "Fig. 5 — LV network module  |  National Energy Digital Twin (NEDT) Ontology")
    return svg(u, W, H, b)


# ═════════════════════════════════════════════════════════════════════════════
# FIG 6 — ATTRIBUTION MODULE
# ═════════════════════════════════════════════════════════════════════════════
def fig6():
    W, H = 900, 540
    u = "6"
    b = ""
    ac = C["red_s"]

    b += box(300, 30, 300, 36, C["red_f"], C["red_s"], "nedt:OverloadedStation")

    # exceedance_kw (left)
    b += parr([(300, 48), (120, 48), (120, 80)], u, ac)
    b += lbl(116, 64, "exceedance", ac, "end")
    b += gbox(30, 80, 182, 44, "exceedance_kw", "peak_kw − designed_kva")

    # hasAttribution (down centre)
    b += arr(450, 66, 450, 80, u, ac)
    b += lbl(456, 74, "hasAttribution", ac, "start")
    b += box(300, 80, 300, 36, C["red_f"], C["red_s"], "nedt:OverloadAttribution")

    # StationTransition (right)
    b += parr([(600, 48), (770, 48), (770, 80)], u, ac)
    b += lbl(684, 42, "hasTransition", ac)
    b += box(680, 80, 190, 52, C["coral_f"], C["coral_s"],
             "nedt:StationTransition", "UnitsMoved_station")

    # DeltaSeries
    b += arr(450, 116, 450, 140, u, ac)
    b += lbl(456, 130, "computesDelta", ac, "start")
    b += box(270, 140, 360, 52, C["red_f"], C["red_s"],
             "nedt:DeltaSeries",
             "(to_profile − from_profile) × units × 1e6")
    b += parr([(360, 192), (310, 216)], u, "#888", dash=True)
    b += parr([(540, 192), (590, 216)], u, "#888", dash=True)
    b += box(200, 216, 164, 44, C["gray_f"], C["gray_s"],
             "t_peak: hour index", "hour_peak = t % 24")
    b += box(502, 216, 196, 44, C["gray_f"], C["gray_s"],
             "contrib_kw_at_tpeak", "abs → rank")

    # ContribRank
    b += arr(450, 260, 450, 284, u, ac)
    b += lbl(456, 275, "ranksBy", ac, "start")
    b += box(290, 284, 320, 36, C["gray_f"], C["gray_s"],
             "nedt:ContribRank  (top_n default 8)")

    # DominantDriver / TopTransitions / RiskTable
    b += parr([(340, 320), (240, 344)], u, ac)
    b += parr([(450, 320), (450, 344)], u, "#888")
    b += parr([(560, 320), (660, 344)], u, ac)
    b += box(140, 344, 200, 52, C["red_f"], C["red_s"],
             "nedt:DominantDriver", "HP|EV|Uncontrolled|V1H|V2H")
    b += box(346, 344, 208, 52, C["gray_f"], C["gray_s"],
             "nedt:TopTransitions", "top1 | top2 | top3")
    b += box(560, 344, 240, 52, C["red_f"], C["red_s"],
             "nedt:RiskTable", "rank | exceedance_kw | top1..top3")

    # ── Legend ────────────────────────────────────────────────────────────
    b += (f'<line x1="30" y1="452" x2="60" y2="452" stroke="{ac}" '
          f'stroke-width="0.9" marker-end="url(#ar{u})"/>')
    b += f'<text class="tl" x="66" y="456" fill="{ac}">object property</text>'
    b += (f'<line x1="200" y1="452" x2="230" y2="452" stroke="#888" '
          f'stroke-width="0.9" stroke-dasharray="5 3" marker-end="url(#ar{u})"/>')
    b += f'<text class="tl" x="236" y="456" fill="#888">subClassOf</text>'
    b += legend(476, [
        (C["red_f"],    C["red_s"],   "overload classes"),
        (C["coral_f"],  C["coral_s"], "transition"),
        (C["gray_f"],   C["gray_s"],  "data/derived"),
    ])
    b += figcap(W, H, "Fig. 6 — Attribution module  |  National Energy Digital Twin (NEDT) Ontology")
    return svg(u, W, H, b)


# ═════════════════════════════════════════════════════════════════════════════
# FIG 7 — HEAT DENSITY MODULE
# ═════════════════════════════════════════════════════════════════════════════
def fig7():
    W, H = 900, 520
    u = "7"
    b = ""
    ac = C["coral_s"]

    b += box(300, 30, 300, 36, C["coral_f"], C["coral_s"], "nedt:HeatDensityGDF")

    # BuildingShare (left)
    b += parr([(300, 48), (120, 48), (120, 80)], u, ac)
    b += lbl(116, 64, "distributedBy", ac, "end")
    b += box(30, 80, 182, 44, C["gray_f"], C["gray_s"],
             "nedt:BuildingShare", "n_bldg / county_total")

    # HullPolygon left-join (right)
    b += parr([(600, 48), (770, 48), (770, 80)], u, ac)
    b += lbl(690, 42, "leftJoin", ac)
    b += box(680, 80, 190, 52, C["blue_f"], C["blue_s"],
             "nedt:HullPolygon", "all Ireland (in_county flag)")

    # HeatFuelComponent (centre)
    b += arr(450, 66, 450, 80, u, ac)
    b += lbl(456, 74, "hasFuelComponent", ac, "start")
    b += box(280, 80, 340, 36, C["coral_f"], C["coral_s"], "nedt:HeatFuelComponent")

    # 4 fuel subclasses — 110px wide, 12px gap, starting x=168
    for i, fl in enumerate(["TotalHeat", "OilHeat", "GasHeat", "HPThermal"]):
        fx = 223 + i * 122
        b += parr([(fx, 116), (fx, 140)], u, "#888", dash=True)
        b += box(168 + i * 122, 140, 110, 30, C["coral_f"], C["coral_s"], fl)

    # HeatDensity
    b += arr(450, 170, 450, 194, u, ac)
    b += lbl(456, 185, "computesDensity", ac, "start")
    b += box(280, 194, 340, 52, C["coral_f"], C["coral_s"],
             "nedt:HeatDensity", "total_gwh / area_km2  →  GWh/km²")

    # 3 column types
    b += parr([(360, 246), (300, 270)], u, "#888", dash=True)
    b += parr([(450, 246), (450, 270)], u, "#888", dash=True)
    b += parr([(540, 246), (600, 270)], u, "#888", dash=True)
    b += box(190, 270, 184, 44, C["amber_f"], C["amber_s"],
             "base_{comp}", "annual_gwh_km2")
    b += box(380, 270, 140, 44, C["gray_f"],  C["gray_s"],
             "scn_{comp}",  "annual_gwh_km2")
    b += box(526, 270, 144, 44, C["gray_f"],  C["gray_s"],
             "delta_{comp}", "scn − base")

    # CountySummary
    b += arr(450, 314, 450, 338, u, ac)
    b += lbl(456, 329, "gdf.attrs[county_summary]", ac, "start")
    b += box(250, 338, 400, 52, C["gray_f"], C["gray_s"],
             "nedt:CountySummary",
             "base_total_gwh | scn_total_gwh | area_km2 | fuel shares")
    b += parr([(320, 390), (280, 412)], u, "#888", dash=True)
    b += parr([(580, 390), (620, 412)], u, "#888", dash=True)
    b += gbox(190, 412, 152, 28, "base_oil/gas/hp_gwh")
    b += gbox(564, 412, 192, 28, "base_density_gwh_km2")

    # ── Legend ────────────────────────────────────────────────────────────
    b += (f'<line x1="30" y1="452" x2="60" y2="452" stroke="{ac}" '
          f'stroke-width="0.9" marker-end="url(#ar{u})"/>')
    b += f'<text class="tl" x="66" y="456" fill="{ac}">object property</text>'
    b += (f'<line x1="200" y1="452" x2="230" y2="452" stroke="#888" '
          f'stroke-width="0.9" stroke-dasharray="5 3" marker-end="url(#ar{u})"/>')
    b += f'<text class="tl" x="236" y="456" fill="#888">subClassOf</text>'
    b += legend(476, [
        (C["coral_f"], C["coral_s"], "heat density"),
        (C["blue_f"],  C["blue_s"],  "LV hull"),
        (C["amber_f"], C["amber_s"], "baseline col"),
        (C["gray_f"],  C["gray_s"],  "derived"),
    ])
    b += figcap(W, H, "Fig. 7 — Heat density module  |  National Energy Digital Twin (NEDT) Ontology")
    return svg(u, W, H, b)


# ═════════════════════════════════════════════════════════════════════════════
# FIG 8 — VISUALISATION / REPORT MODULE
# ═════════════════════════════════════════════════════════════════════════════
def fig8():
    W, H = 900, 540
    u = "8"
    b = ""
    ac = C["purple_s"]

    b += box(320, 30, 260, 36, C["purple_f"], C["purple_s"], "nedt:FigCollector")

    # ── Left: DeckGLMap ───────────────────────────────────────────────────
    b += parr([(320, 48), (140, 48), (140, 80)], u, ac)
    b += lbl(136, 64, "hasDeckMap", ac, "end")
    b += box(40, 80, 200, 36, C["purple_f"], C["purple_s"], "nedt:DeckGLMap")
    b += parr([(100, 116), ( 80, 140)], u, "#888", dash=True)
    b += parr([(180, 116), (200, 140)], u, "#888", dash=True)
    b += box( 24, 140, 112, 44, C["purple_f"], C["purple_s"], "Categorical",  "status, driver")
    b += box(142, 140, 116, 44, C["purple_f"], C["purple_s"], "Choropleth",   "GWh/km², ΔkW")
    b += arr(140, 184, 140, 208, u, ac)
    b += lbl(146, 198, "usesBasemap", ac, "start")
    b += gbox(40, 208, 200, 44, "CartoDB Positron", "(no token required)")

    # ── Centre: ReportSection ─────────────────────────────────────────────
    b += arr(450, 66, 450, 80, u, ac)
    b += lbl(456, 74, "hasSections", ac, "start")
    b += box(280, 80, 340, 36, C["purple_f"], C["purple_s"], "nedt:ReportSection")
    b += parr([(360, 116), (310, 140)], u, "#888", dash=True)
    b += parr([(450, 116), (450, 140)], u, "#888", dash=True)
    b += parr([(540, 116), (590, 140)], u, "#888", dash=True)
    b += gbox(244, 140, 122, 28, "Heading / Table")
    b += box(376, 140, 148, 28, C["purple_f"], C["purple_s"], "DeckGL section")
    b += box(530, 140, 140, 28, C["purple_f"], C["purple_s"], "ChartJS section")

    # 9 scenario sections
    b += arr(450, 168, 450, 192, u, ac)
    b += lbl(456, 183, "scenarioSections", ac, "start")
    b += box(170, 192, 520, 52, C["gray_f"], C["gray_s"],
             "9-section scenario report",
             "HP moves | EV modes | Ledger | Energy+CO₂ | Heat | Worsened | LV | Overload | Diagnostic")

    # ColourPalette
    b += parr([(300, 244), (200, 268)], u, ac)
    b += lbl(196, 264, "usePalette", ac, "end")
    b += box(30, 268, 320, 36, C["purple_f"], C["purple_s"], "nedt:ColourPalette")
    b += parr([(100, 304), ( 70, 328)], u, "#888", dash=True)
    b += parr([(190, 304), (190, 328)], u, "#888", dash=True)
    b += parr([(280, 304), (310, 328)], u, "#888", dash=True)
    b += gbox( 22, 328,  96, 28, "Sequential")
    b += gbox(142, 328,  96, 28, "Diverging")
    b += gbox(262, 328,  96, 28, "Categorical")
    b += lbl(190, 374, "PAL_YLORD | PAL_RDBU | PAL_STATUS | PAL_DRIVER", "#888")

    # HTMLReport
    b += arr(450, 244, 450, 268, u, ac)
    b += lbl(456, 259, "open()", ac, "start")
    b += box(350, 268, 200, 44, C["gray_f"], C["gray_s"],
             "nedt:HTMLReport", "tempfile → webbrowser")
    b += parr([(400, 312), (360, 336)], u, "#888", dash=True)
    b += parr([(500, 312), (540, 336)], u, "#888", dash=True)
    b += gbox(280, 336, 152, 28, "deck.gl CDN (WebGL)")
    b += gbox(440, 336, 152, 28, "Chart.js 4 CDN")

    # ── Right: ChartJSChart ───────────────────────────────────────────────
    b += parr([(580, 48), (770, 48), (770, 80)], u, ac)
    b += lbl(684, 42, "hasChart", ac)
    b += box(680,  80, 190, 36, C["purple_f"], C["purple_s"], "nedt:ChartJSChart")
    b += parr([(730, 116), (700, 140)], u, "#888", dash=True)
    b += parr([(775, 116), (775, 140)], u, "#888", dash=True)
    b += parr([(820, 116), (850, 140)], u, "#888", dash=True)
    b += box(658, 140,  90, 44, C["purple_f"], C["purple_s"], "TimeSeries", "8760 h")
    b += box(754, 140,  80, 44, C["purple_f"], C["purple_s"], "Multipanel", "6-panel CO₂")
    b += box(840, 140,  52, 44, C["purple_f"], C["purple_s"], "Bar",        "fuel mix")
    b += arr(775, 184, 775, 208, u, ac)
    b += lbl(781, 198, "downsample", ac, "start")
    b += gbox(680, 208, 190, 28, "every 168th pt (weekly)")

    # ── Legend ────────────────────────────────────────────────────────────
    b += (f'<line x1="30" y1="416" x2="60" y2="416" stroke="{ac}" '
          f'stroke-width="0.9" marker-end="url(#ar{u})"/>')
    b += f'<text class="tl" x="66" y="420" fill="{ac}">object property</text>'
    b += (f'<line x1="200" y1="416" x2="230" y2="416" stroke="#888" '
          f'stroke-width="0.9" stroke-dasharray="5 3" marker-end="url(#ar{u})"/>')
    b += f'<text class="tl" x="236" y="420" fill="#888">subClassOf</text>'
    b += legend(440, [
        (C["purple_f"], C["purple_s"], "report classes"),
        (C["gray_f"],   C["gray_s"],   "data/config"),
    ])
    b += figcap(W, H, "Fig. 8 — Visualisation / report module  |  National Energy Digital Twin (NEDT) Ontology")
    return svg(u, W, H, b)


# ═════════════════════════════════════════════════════════════════════════════
# FIGURE REGISTRY
# ─────────────────────────────────────────────────────────────────────────────
# To add a new diagram:
#   1. Write  def fig9(): ...  above
#   2. Add an entry here:  "fig9_my_module": (fig9, "Fig. 9 — My module", "description")
#   3. Run:  python3 gen_diagrams.py
# ═════════════════════════════════════════════════════════════════════════════
FIGS = {
    "fig0_conceptual_model": (
        fig0,
        "Fig. 0 — Conceptual model overview",
        "All 8 modules and their interconnections: KPI, Archetype, Scenario, "
        "Energy Load, LV Network, Attribution, Heat Density, Visualisation",
    ),
    "fig1_kpi_module": (
        fig1,
        "Fig. 1 — KPI module",
        "Stakeholders → goals → Strategic / Tactical / Operational KPI hierarchy "
        "→ KPICalculation → DatumSource → KPIValue with units and evaluation interval",
    ),
    "fig2_archetype_module": (
        fig2,
        "Fig. 2 — Archetype module",
        "Five descriptor columns, AC_ORIG/EV/NOEV counts, BER & heating system "
        "canonical mappings, ProfileKey, BERFallbackRule, TransitionKey",
    ),
    "fig3_scenario_module": (
        fig3,
        "Fig. 3 — Scenario module",
        "HPScenario, EVScenario, TransitionLedger, LargestRemainderAlloc, "
        "EVMode (Uncontrolled/V1H/V2H), Tariff (Tier3/4/Dynamic), NexsysProfile",
    ),
    "fig4_energy_load_module": (
        fig4,
        "Fig. 4 — Energy load module",
        "ElectricLoad (two-component EV formula: NoEV base + Nexsys charging), "
        "HeatLoad (oil/gas/HP split), CO2Emission factors, LoadCalculation",
    ),
    "fig5_lv_network_module": (
        fig5,
        "Fig. 5 — LV network module",
        "LVStation, HullPolygon (area_km2 via EPSG:2157), StationCapacity "
        "(designed_kva, default 200 kVA), StationLoad, Utilisation, "
        "5-band UtilisationStatus, BuildingShare, in_county flag",
    ),
    "fig6_attribution_module": (
        fig6,
        "Fig. 6 — Attribution module",
        "OverloadedStation, DeltaSeries computation, t_peak / hour_peak, "
        "contrib_kw_at_tpeak ranking, DominantDriver, TopTransitions, RiskTable",
    ),
    "fig7_heat_density_module": (
        fig7,
        "Fig. 7 — Heat density module",
        "HeatDensityGDF, four HeatFuelComponent subclasses (Total/Oil/Gas/HP), "
        "HeatDensity (GWh/km²), baseline/scenario/delta columns, CountySummary",
    ),
    "fig8_visualisation_module": (
        fig8,
        "Fig. 8 — Visualisation / report module",
        "FigCollector, DeckGLMap (Categorical + Choropleth), ChartJSChart "
        "(TimeSeries / Multipanel / Bar), 9 report sections, ColourPalette, HTMLReport",
    ),
}


# ═════════════════════════════════════════════════════════════════════════════
# HTML ASSEMBLY
# ═════════════════════════════════════════════════════════════════════════════
HTML_HEADER = """\
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>National Energy Digital Twin (NEDT) — Ontology Diagrams</title>
<style>
body { font-family: Arial, sans-serif; background: #f0f2f5; margin: 0; padding: 24px; }
h1   { color: #2c3e50; text-align: center; font-size: 20px; margin-bottom: 6px; }
.sub { text-align: center; color: #666; font-size: 13px; margin-bottom: 28px; }
.dl-all {
  display: block; width: 280px; margin: 0 auto 32px; padding: 13px;
  background: #534AB7; color: white; border: none; border-radius: 8px;
  font-size: 14px; font-weight: 700; cursor: pointer; text-align: center;
}
.dl-all:hover { background: #3C3489; }
.card {
  background: white; border-radius: 10px; padding: 18px 22px;
  margin: 0 auto 24px; max-width: 940px;
  box-shadow: 0 2px 8px rgba(0,0,0,.10);
}
.card h2  { color: #2c3e50; font-size: 14px; margin: 0 0 3px; }
.card p   { color: #666; font-size: 12px; margin: 0 0 10px; }
.card svg { display: block; width: 100%; border: 1px solid #e8e8e8;
            border-radius: 6px; background: white; }
.btns { display: flex; gap: 10px; margin-top: 10px; }
.btn  { padding: 7px 18px; border: none; border-radius: 6px;
        cursor: pointer; font-size: 13px; font-weight: 700; color: white; }
.bp { background: #185FA5; } .bp:hover { background: #0C447C; }
.bs { background: #0F6E56; } .bs:hover { background: #085041; }
</style>
</head>
<body>
<h1>National Energy Digital Twin (NEDT) — Ontology Diagrams</h1>
<p class="sub">
  9 module diagrams modelled on Li et al. (2019) EM-KPI ontology style.<br>
  Click <strong>Download PNG</strong> (3&times; high-res, ~2700&times;1600px) or
  <strong>Download SVG</strong> (vector, editable in Illustrator / Inkscape).
</p>
<button class="dl-all" onclick="dlAll()">&#8595; Download All 9 PNGs</button>
"""

HTML_SCRIPT = """\
<script>
function getSVG(id) {
  // SVG ids follow the pattern svg0, svg1 … svg8
  return document.getElementById('svg' + id.replace(/^fig(\\d).*/, '$1'));
}

function dlPNG(id) {
  const svg = getSVG(id);
  if (!svg) { alert('SVG not found: ' + id); return; }
  const vb    = svg.getAttribute('viewBox').split(' ').map(Number);
  const scale = 3;
  const W = vb[2] * scale;
  const H = vb[3] * scale;
  const canvas = document.createElement('canvas');
  canvas.width  = W;
  canvas.height = H;
  const ctx = canvas.getContext('2d');
  ctx.fillStyle = 'white';
  ctx.fillRect(0, 0, W, H);
  // Clone the SVG and give it explicit pixel dimensions so the browser
  // renders it at the right size when drawn onto the canvas.
  const clone = svg.cloneNode(true);
  clone.setAttribute('width',  vb[2]);
  clone.setAttribute('height', vb[3]);
  const str  = new XMLSerializer().serializeToString(clone);
  const blob = new Blob([str], { type: 'image/svg+xml;charset=utf-8' });
  const url  = URL.createObjectURL(blob);
  const img  = new Image();
  img.onload = () => {
    ctx.drawImage(img, 0, 0, W, H);
    URL.revokeObjectURL(url);
    const a    = document.createElement('a');
    a.download = id + '.png';
    a.href     = canvas.toDataURL('image/png');
    a.click();
  };
  img.onerror = () => {
    URL.revokeObjectURL(url);
    alert('PNG render failed for ' + id + ' — try SVG instead.');
  };
  img.src = url;
}

function dlSVG(id) {
  const svg = getSVG(id);
  if (!svg) return;
  const vb    = svg.getAttribute('viewBox').split(' ').map(Number);
  const clone = svg.cloneNode(true);
  clone.setAttribute('width',  vb[2]);
  clone.setAttribute('height', vb[3]);
  const str  = '<?xml version="1.0" encoding="UTF-8"?>\\n'
             + new XMLSerializer().serializeToString(clone);
  const blob = new Blob([str], { type: 'image/svg+xml' });
  const a    = document.createElement('a');
  a.download = id + '.svg';
  a.href     = URL.createObjectURL(blob);
  a.click();
}

const IDS = FIGURE_IDS_PLACEHOLDER;

function dlAll() {
  // Stagger downloads so the browser doesn't suppress them
  IDS.forEach((id, i) => setTimeout(() => dlPNG(id), i * 900));
}
</script>
</body>
</html>
"""


# ═════════════════════════════════════════════════════════════════════════════
# MAIN
# ═════════════════════════════════════════════════════════════════════════════
def main():
    parts = [HTML_HEADER]

    for fig_id, (fn, title, desc) in FIGS.items():
        print(f"  Generating {fig_id} …")
        svg_content = fn()
        parts.append(
            f'<div class="card">\n'
            f'<h2>{title}</h2>\n'
            f'<p>{desc}</p>\n'
            f'{svg_content}\n'
            f'<div class="btns">\n'
            f'  <button class="btn bp" onclick="dlPNG(\'{fig_id}\')">Download PNG</button>\n'
            f'  <button class="btn bs" onclick="dlSVG(\'{fig_id}\')">Download SVG</button>\n'
            f'</div>\n'
            f'</div>\n'
        )

    # Inject the figure IDs into the JS
    id_list = str(list(FIGS.keys()))
    script  = HTML_SCRIPT.replace("FIGURE_IDS_PLACEHOLDER", id_list)
    parts.append(script)

    html = "".join(parts)
    OUT_FILE.write_text(html, encoding="utf-8")

    size_kb = OUT_FILE.stat().st_size // 1024
    print(f"\nDone!  Written → {OUT_FILE}  ({size_kb} KB)")

    if AUTO_OPEN:
        webbrowser.open(OUT_FILE.as_uri())
        print("Opened in your default browser.")
    else:
        print(f"Open manually:  open '{OUT_FILE}'")


if __name__ == "__main__":
    main()

  Generating fig0_conceptual_model …
  Generating fig1_kpi_module …
  Generating fig2_archetype_module …
  Generating fig3_scenario_module …
  Generating fig4_energy_load_module …
  Generating fig5_lv_network_module …
  Generating fig6_attribution_module …
  Generating fig7_heat_density_module …
  Generating fig8_visualisation_module …

Done!  Written → /Users/divyanshusood/Documents/DT_Model/ontology_diagrams.html  (95 KB)
Opened in your default browser.
